# Evaluating agents with Inspect AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meta-pytorch/OpenEnv/blob/main/examples/evaluation_inspect.ipynb)

After training a model in an OpenEnv environment, you need to measure how it actually performs on a held-out set of episodes. This notebook shows how to use [Inspect AI](https://inspect.aisi.org.uk/) through OpenEnv's `InspectAIHarness` to run structured, reproducible evals.

**What you'll build:**
- An Inspect AI `Task` whose solver calls an OpenEnv environment
- A custom scorer that grades the env's response
- A reproducible eval run tracked as an `EvalResult`

**Model providers supported:** OpenAI or Anthropic via their respective APIs.

---

## 1. Install dependencies

In [ ]:
!pip install -q "inspect-ai>=0.3.0"
!pip install -q "openenv-core @ git+https://github.com/meta-pytorch/OpenEnv.git"

---

## 2. Set your model provider

Uncomment exactly one option. All three feed into the same task and harness in
Section 4 — no other cells need to change.

In [ ]:
import getpass, os

# --- Option A: OpenAI ---
os.environ.setdefault("OPENAI_API_KEY", getpass.getpass("OpenAI API key: "))
MODEL = "openai/gpt-5-mini"

# --- Option B: Anthropic ---
# os.environ.setdefault("ANTHROPIC_API_KEY", getpass.getpass("Anthropic API key: "))
# MODEL = "anthropic/claude-haiku-4-5-20251001"

# --- Option C: local transformers model (no API key needed) ---
# Requires a GPU for reasonable speed. Omit 'temperature' from eval_parameters below.
# !pip install -U transformers
# MODEL = "hf/Qwen/Qwen3.5-0.8B"
# Use a local checkpoint path to skip the download:
# MODEL = "hf/./outputs/my-trained-model"

---

## 3. How Inspect AI and OpenEnv fit together

- **OpenEnv** provides the environment: `reset()`, tool calls, reward.
- **Inspect AI** provides the eval infrastructure: datasets, solvers, scorers, and structured logs.
- **`InspectAIHarness`** is the bridge — it wraps `inspect_ai.eval()` so runs are tracked
  with `EvalConfig` / `EvalResult`.

An Inspect AI `Task` has three parts:
1. **Dataset** — samples with `input` and `target` fields
2. **Solver** — calls the model and the env for each sample
3. **Scorer** — grades the result

> **Note on the env client:** `echo_env` is a pure MCP environment. Interact with it via
> `MCPToolClient` and `call_tool("echo_message", ...)`. For non-MCP environments, use
> `GenericEnvClient` instead.

---

## 4. Define the task

The solver calls Inspect AI's `generate()` to get the model's output, then sends it
to the environment. The dataset, scorer, and harness are identical for both providers.

In [ ]:
import asyncio

from inspect_ai import Task, task
from inspect_ai.dataset import Sample
from inspect_ai.scorer import CORRECT, INCORRECT, Score, Target, accuracy, scorer
from inspect_ai.solver import Generate, TaskState, solver

from openenv.core import MCPToolClient

ECHO_ENV_URL = "https://openenv-echo-env.hf.space"

# Limit concurrent env connections to match the server's MAX_CONCURRENT_ENVS.
_env_sem = asyncio.Semaphore(1)  # increase if your Space supports more sessions


@task
def openenv_echo_eval(base_url: str = ECHO_ENV_URL):
    return Task(
        dataset=[
            Sample(input="Repeat exactly: hello world", target="hello world"),
            Sample(input="Repeat exactly: inspect ai", target="inspect ai"),
            Sample(input="Repeat exactly: openenv eval", target="openenv eval"),
            Sample(input="Repeat exactly: reinforcement learning", target="reinforcement learning"),
            Sample(input="Repeat exactly: hugging face", target="hugging face"),
        ],
        solver=echo_env_solver(base_url=base_url),
        scorer=echo_scorer(),
    )


@solver
def echo_env_solver(base_url: str):
    """Ask the model to repeat the phrase, then echo it through the env."""

    async def solve(state: TaskState, generate: Generate) -> TaskState:
        state = await generate(state)
        model_output = state.output.completion.strip()

        async with _env_sem:  # one env connection at a time
            env = MCPToolClient(base_url=base_url)
            try:
                await env.reset()
                echoed = await env.call_tool("echo_message", message=model_output)
                state.metadata["echoed"] = str(echoed) if echoed is not None else ""
            finally:
                await env.close()

        return state

    return solve


@scorer(metrics=[accuracy()])
def echo_scorer():
    """CORRECT if the env echoed back exactly what the target phrase was."""

    async def score(state: TaskState, target: Target) -> Score:
        echoed = state.metadata.get("echoed", "").strip()
        expected = target.text.strip()
        return Score(
            value=CORRECT if echoed == expected else INCORRECT,
            explanation=f"Env echoed {echoed!r}, expected {expected!r}",
        )

    return score

---

## 5. Run the eval with `InspectAIHarness`

`EvalConfig` captures everything needed to reproduce the run — including which model was used.

In [ ]:
import inspect_ai
import openenv

from openenv.core.evals import EvalConfig, EvalResult, InspectAIHarness

harness = InspectAIHarness(log_dir="./eval-logs")

config = EvalConfig(
    harness_name="InspectAIHarness",
    harness_version=inspect_ai.__version__,
    library_versions={"openenv": openenv.__version__},
    dataset="openenv_echo_eval",
    eval_parameters={
        "model": MODEL,
        "task": openenv_echo_eval(base_url=ECHO_ENV_URL),
        # temperature is supported for API providers (Options A/B).
        # Omit it for local transformers models (Option C).
        "temperature": 0.0,
    },
)

result: EvalResult = harness.run_from_config(config)
print(result.scores)
# {'accuracy': 1.0}

---

## 6. Inspect the results

`EvalResult` carries both the config and the scores, making it easy to log, compare across
runs, or push to the Hub.

In [ ]:
import json

class _StrFallback(json.JSONEncoder):
    def default(self, o):
        return str(o)

print(json.dumps(result.model_dump(), indent=2, cls=_StrFallback))

In [ ]:
# Inspect AI also writes detailed logs to ./eval-logs/
# !inspect view --log-dir ./eval-logs

---

## 7. Using a task file instead of a task object

Inspect AI tasks can also be defined in standalone `.py` files and referenced
by path. This is useful for CI pipelines where the task definition lives in
the repo and the harness is called from a script:

In [ ]:
# tasks/echo_eval.py  (contains the @task definition above)

result = harness.run_from_config(EvalConfig(
    harness_name="InspectAIHarness",
    harness_version=inspect_ai.__version__,
    library_versions={"openenv": openenv.__version__},
    dataset="tasks/echo_eval.py@openenv_echo_eval",
    eval_parameters={
        "model": "openai/gpt-5-mini",
        "task": "tasks/echo_eval.py@openenv_echo_eval",
    },
))

---

## 8. Adapt to your own environment

1. **Dataset** — held-out episodes from your env. Each `Sample` needs `input` and `target`.
2. **Solver** — call your env's MCP tool in place of `echo_message`.
3. **Scorer** — compare the tool result against the target, or use the env's reward signal.

In [ ]:
import asyncio

from inspect_ai.solver import Generate, TaskState, solver
from openenv.core import MCPToolClient

_env_sem = asyncio.Semaphore(1)  # raise if your Space supports more sessions


@solver
def my_env_solver(base_url: str):
    async def solve(state: TaskState, generate: Generate) -> TaskState:
        state = await generate(state)
        model_output = state.output.completion.strip()

        async with _env_sem:
            env = MCPToolClient(base_url=base_url)
            try:
                await env.reset()
                result = await env.call_tool("your_tool_name", message=model_output)
                state.metadata["env_result"] = result
            finally:
                await env.close()
        return state

    return solve

---

## Next steps

- **[Wordle GRPO tutorial](https://meta-pytorch.org/OpenEnv/tutorials/wordle-grpo.html)** — train a model you can then evaluate with this notebook
- **[Rubrics tutorial](https://meta-pytorch.org/OpenEnv/tutorials/rubrics.html)** — define composable reward functions inside the environment
- **[Inspect AI documentation](https://inspect.aisi.org.uk/)** — full reference for tasks, solvers, scorers, and the log viewer